# 02 - Build

NB03 (performans) ve NB04 (kontrol mekanizmasi) ayni turetmeyi iki kere
yapmasin diye episode/regime segmentasyonu, state, action sinifi, T0 ve
event tespiti burada bir kez uretilir.

**Iki ayri birim var, karistirilmamali:**

| Birim | Tanim | Ne icin |
|---|---|---|
| Episode | reset'ten reset'e | T0, sure (Ludolph) |
| Regime run | kuadran dizisi, theta*omega isaret degistirince yeni run | Safe/Saved/Failed (Park) |

Girdi: NB01'in yazdigi `samples_clean.parquet` ve `trials_clean.parquet`.

In [1]:
%pip install -q pyyaml pandas numpy pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

# Aktif veri seti. config.yaml -> datasets. Setler birbirine karismaz:
# her set kendi data/<dataset>/{raw,interim,processed} agacinda durur.
DATASET = "pilot2"

ANALYSIS_ROOT = Path.cwd()
for _ in range(4):
    if (ANALYSIS_ROOT / "config.yaml").exists():
        break
    ANALYSIS_ROOT = ANALYSIS_ROOT.parent
sys.path.insert(0, str(ANALYSIS_ROOT))

from src.physics import params_from_config, compute_T0, verify_model
from src.build import build_all, validate_T0_freefall

from src.dataset import load_config, dirs

config, ANALYSIS_ROOT = load_config(DATASET, ANALYSIS_ROOT)
RAW_DIR, INTERIM_DIR, PROCESSED_DIR = dirs(config, ANALYSIS_ROOT)
print(f"veri seti: {config['dataset']}  ({config['dataset_label']})")

pd.set_option("display.max_columns", None)

df_samples = pd.read_parquet(INTERIM_DIR / "samples_clean.parquet")
df_trials = pd.read_parquet(INTERIM_DIR / "trials_clean.parquet")
print(f"Sample: {len(df_samples):,}   Trial: {len(df_trials)}")

veri seti: pilot2  (Pilot 2 -- 9 katilimci, 2-3 Eylul 2026)
Sample: 623,700   Trial: 477


## 1. Fizik modeli dogrulamasi

T0 hesabi bir modele dayaniyor. Model varsayilmiyor: kayitli
`applied_force_n` ile simule edilen acisal ivme, gozlenen ivmeyle
karsilastiriliyor. Korelasyon dusukse T0'a guvenilmez.

In [3]:
p = params_from_config(config)
print("Parametreler:")
for k, v in p.items():
    print(f"  {k:18} {v}")

ver = verify_model(df_samples, p)
if ver.empty:
    print("\nDogrulanamadi.")
else:
    lo, hi = ver.corr_alpha.min(), ver.corr_alpha.max()
    print(f"\nModel dogrulama: {len(ver)} parca, korelasyon {lo:.4f} - {hi:.4f}")
    if lo < 0.95:
        print("UYARI: korelasyon dusuk, T0 guvenilir degil.")
    display(ver)

print("\nT0(theta0):")
for a in [0.5, 1, 2, 3, 5, 7, 7.5]:
    print(f"  {a:4.1f} deg -> {compute_T0(a, p):.2f} s")

Parametreler:
  m_c                0.4
  m_p                0.08
  l                  0.5
  g                  1.0
  k                  1.3333333333333333
  angle_limit_deg    60.0
  track_limit_m      5.0
  dt                 0.016666666666666666
  max_time_s         60.0
  round_decimals     4

Model dogrulama: 8 parca, korelasyon 0.9943 - 0.9979


,participant_id,trial_id,n,corr_alpha,rms_alpha
0,P001,T002,317,0.9965,0.1968
1,P001,T011,309,0.9958,0.1506
2,P001,T014,371,0.9943,0.4005
3,P001,T017,297,0.9957,0.3516
4,P001,T018,375,0.9969,0.2953
5,P001,T019,347,0.9979,0.3048
6,P001,T020,385,0.9944,0.2490
7,P001,T021,296,0.9952,0.1597



T0(theta0):
   0.5 deg -> 4.23 s
   1.0 deg -> 3.70 s
   2.0 deg -> 3.18 s
   3.0 deg -> 2.87 s
   5.0 deg -> 2.48 s
   7.0 deg -> 2.22 s
   7.5 deg -> 2.17 s


## 2. Turetme

`build_all` sirayla: state (kuadran) -> action sinifi -> episode ->
regime run -> girdi olayi.

In [4]:
df_built, episodes, regimes, input_events = build_all(df_samples, df_trials, config)

print(f"sample       {df_built.shape}")
print(f"episode      {episodes.shape}")
print(f"regime       {regimes.shape}")
print(f"input_event  {input_events.shape}")

sample       (623700, 31)
episode      (1332, 25)
regime       (15133, 20)
input_event  (28488, 12)


## 2b. T0'in ampirik dogrulamasi

Katilimcinin hic girdi vermedigi ve aci limitiyle biten episode'lar tanim
geregi serbest dususturler. Bu episodelarda gozlenen sure T0'a **esit**
olmali. Fizik modeli, RK4 adimi, T0 hesabi ve episode segmentasyonu
zincirinin tamamini tek seferde test eder.

In [5]:
free = validate_T0_freefall(df_built, episodes)

if free.empty:
    print("Girdisiz, aci limitiyle biten episode bulunamadi - test yapilamadi.")
else:
    n = free.attrs["n"]
    dev = free.attrs["max_dev"]
    print(f"Serbest dusus episode'u: {n}")
    print(f"duration/T0 ortalama: {free.attrs['mean_ratio']:.4f}")
    print(f"1.0'dan max sapma:    {dev:.4f}")
    print("SONUC:", "T0 dogrulandi" if dev < 0.02 else "UYARI - T0 tutmuyor")
    display(free.round(4))

Serbest dusus episode'u: 1
duration/T0 ortalama: 1.0000
1.0'dan max sapma:    0.0000
SONUC: T0 dogrulandi


,participant_id,trial_id,episode,theta0_deg,omega0_deg_s,duration_s,T0_s,duration_over_T0
0,P004,T043,0.0,-6.4963,-0.1848,2.2834,2.2833,1.0


## 3. State ve action dagilimi

Isaret konvansiyonu veriden dogrulandi: duzeltici kuvvet theta ile
**ayni** isaretli. Park'in metnindekinin tersi, cunku onun VIP'inde
joystick dogrudan acisal ivme veriyor, bizde kuvvet cart'a gidiyor.

In [6]:
act = df_built[df_built["phase"] == "active"]

print("Kuadran (%):")
q = act["falling"].map({True: "fall", False: "safe"}).value_counts(normalize=True)
print((q * 100).round(1).to_string())

print()
print("Action sinifi (%):")
vc = (act["action"].value_counts(normalize=True) * 100).round(2)
display(vc.to_frame("pct"))

x = act[act["action"] == "X"]
if len(x):
    print(f"X ({len(x)} ornek, %{100 * len(x) / len(act):.2f}) sebepleri:")
    print(x["action_excluded_reason"].value_counts().to_string())

print()
print("Katilimci basina action (%):")
tab = act.groupby(["participant_id", "action"]).size().unstack(fill_value=0)
display((100 * tab.div(tab.sum(axis=1), axis=0)).round(1))

Kuadran (%):
falling
fall    61.8
safe    38.2

Action sinifi (%):


,pct
action,
I,78.68
CR,19.23
D,1.66
A,0.42
X,0.01


X (81 ornek, %0.01) sebepleri:
action_excluded_reason
transient_neutral    80
degenerate_sign       1

Katilimci basina action (%):


action,A,CR,D,I,X
participant_id,,,,,
P001,0.2,26.8,4.1,68.9,0.0
P002,0.5,21.7,0.8,77.0,0.0
P003,0.2,22.3,0.3,77.3,0.0
P004,0.4,25.8,2.1,71.8,0.0
P005,0.4,15.5,1.4,82.6,0.0
P006,0.4,15.1,2.3,82.2,0.0
P007,0.4,18.1,0.9,80.5,0.0
P008,0.6,13.1,0.6,85.6,0.0
P009,0.7,14.6,2.4,82.2,0.1


## 4. Episode

Reset'ten reset'e. `censored` = dususle degil trial bitisiyle sona erdi,
yani ne kadar daha dayanacagi bilinmiyor.

`fall_cause` iki degerli: **angle** (pole +-60 dereceye vardi) veya
**track** (cart +-5 m ray sinirina carpti). Ikisi ayri basarisizlik
turu; ray kaynakli dususlerde pole cogu zaman dik duruyor.

In [7]:
print(f"Episode: {len(episodes)}")
print(f"  dususle biten : {int(episodes.ended_in_fall.sum())}")
print(f"  sansurlu      : {int(episodes.censored.sum())}")
print()
print("Dusus sebebi:")
display(episodes.fall_cause.value_counts(dropna=False).to_frame("n"))

print("Sebep basina dusus anindaki max |theta|:")
fall_ep = episodes[episodes.ended_in_fall]
display(fall_ep.groupby("fall_cause").max_abs_theta_deg.describe().round(2))

print()
print("Episode sure ve T0:")
display(episodes[["duration_s", "theta0_deg", "T0_s",
                  "duration_over_T0"]].describe().round(3))

n_nan = int(episodes.T0_s.isna().sum())
if n_nan:
    print(f"UYARI: {n_nan} episode'da T0 hesaplanamadi (theta0 ~ 0).")

Episode: 1332
  dususle biten : 855
  sansurlu      : 477

Dusus sebebi:


,n
fall_cause,
angle,752
NaN,477
track,103


Sebep basina dusus anindaki max |theta|:


,count,mean,std,min,25%,50%,75%,max
fall_cause,,,,,,,,
angle,752.0,61.10,0.94,60.00,60.36,60.82,61.60,64.57
track,103.0,41.15,13.59,9.98,32.26,44.80,52.02,59.95



Episode sure ve T0:


,duration_s,theta0_deg,T0_s,duration_over_T0
count,1332.000,1332.000,1332.000,1332.000
mean,7.162,-0.043,2.954,2.540
std,6.146,4.249,0.759,2.282
min,0.050,-7.501,2.167,0.016
25%,2.363,-3.708,2.400,0.820
50%,4.967,0.082,2.717,1.709
75%,10.196,3.528,3.254,3.516
max,20.000,7.495,6.817,9.231


## 5. Regime run

Park'in rejimleri. `Failed` sadece aci kaynakli dususler icin -- Park ile
karsilastirilabilir olan bu. `TrackLoss` ray kaybi, Park'ta karsiligi yok,
Park karsilastirmalarindan cikarilmali.

In [8]:
print(f"Regime run: {len(regimes)}")
n_ep = regimes.groupby(["participant_id", "trial_id", "episode"]).ngroups
print(f"Episode basina ortalama run: {len(regimes) / n_ep:.1f}")
print()
tab = regimes.regime.value_counts().to_frame("n")
tab["pct"] = (100 * tab.n / len(regimes)).round(1)
display(tab)

print("Rejim basina sure ve max |theta|:")
display(regimes.groupby("regime")[["duration_s", "max_abs_theta_deg"]].mean().round(2))

print()
print("Rejim basina action dagilimi (%):")
display(regimes.groupby("regime")[["pct_I", "pct_CR", "pct_A", "pct_D"]].mean().round(1))

min_n = config["build"]["regime_min_samples"]
short = regimes[regimes.n_samples < min_n]
print()
print(f"Cok kisa run (< {min_n} ornek): {len(short)}")

Regime run: 15133
Episode basina ortalama run: 11.4



,n,pct
regime,,
Saved,7000,46.3
Safe,6998,46.2
Failed,752,5.0
censored,280,1.9
TrackLoss,103,0.7


Rejim basina sure ve max |theta|:


,duration_s,max_abs_theta_deg
regime,,
Failed,0.81,61.10
Safe,0.52,15.18
Saved,0.73,15.27
TrackLoss,0.44,22.83
censored,0.60,11.39



Rejim basina action dagilimi (%):


,pct_I,pct_CR,pct_A,pct_D
regime,,,,
Failed,57.4,19.4,0.0,23.0
Safe,64.8,34.0,1.2,0.0
Saved,77.6,21.4,0.0,0.9
TrackLoss,60.5,28.0,0.5,10.9
censored,90.3,6.0,0.0,3.6



Cok kisa run (< 2 ornek): 122


## 6. Girdi olaylari

`onset` notr banddan cikis, `offset` banda donus, `reversal` kuvvet yon
degistirme, `fall` dusus.

**Bunlar Ludolph'un event'leri DEGIL.** Ludolph'un action timing analizinde
olay DURUM tarafinda tanimli: pole belirli bir tamsayi aciyi duserken
geciyor. Oradaki olcum, o ana ortalanmis kuvvet segmentlerinin ortalamasinin
sifir gecisi -- buradaki `reversal` sayimi degil. Ayrim ve gerekce:
`Documentation/Yontem/05_Action_Timing.md`.

Buradaki olaylar tanimlayici istatistik ve QC icin. `fall` sayisinin
Unity'nin `fall_count`'uyla karsilastirilmasi bagimsiz bir dogrulama.

Olaylar episode icinde araniyor. Reset satirlarinda `applied_force_n`
sifira zorlanip `input_applied` son degerinde kaldigi icin, parcalari uc
uca eklemek sahte zero-crossing uretirdi.

In [9]:
display(input_events.event.value_counts().to_frame("n"))

n_fall_ev = int((input_events.event == "fall").sum())
n_fall_ts = int(df_trials.fall_count.sum())
status = "TUTUYOR" if n_fall_ev == n_fall_ts else "TUTMUYOR"
print(f"fall event {n_fall_ev} vs trial_summary fall_count {n_fall_ts} -> {status}")

print()
print("Trial basina girdi olayi (measurement, qc_pass):")
ok = df_trials[(df_trials.practice == 0) & df_trials.qc_pass]
ok = ok[["participant_id", "trial_id"]]
ev_ok = input_events.merge(ok, on=["participant_id", "trial_id"])
per = ev_ok.groupby(["participant_id", "trial_id"]).event.value_counts().unstack(fill_value=0)
display(per.mean().round(2).to_frame("trial basina ortalama"))

,n
event,
onset,10923
offset,10600
reversal,6110
fall,855


fall event 855 vs trial_summary fall_count 855 -> TUTUYOR

Trial basina girdi olayi (measurement, qc_pass):


,trial basina ortalama
event,
fall,1.63
offset,22.12
onset,22.78
reversal,12.77


## 7. Ornek: bir episode'un run'lari

Segmentasyonun dogru calistigini gozle dogrulamak icin.

In [10]:
pid = episodes.participant_id.iloc[0]
sel = episodes[(episodes.participant_id == pid) & episodes.ended_in_fall]
if len(sel):
    tid = sel.trial_id.iloc[0]
    epi = sel.episode.iloc[0]
    print(f"{pid} / {tid} / episode {epi}")
    display(sel.iloc[[0]][["duration_s", "theta0_deg", "T0_s",
                           "duration_over_T0", "fall_cause"]].round(3))
    r = regimes[(regimes.participant_id == pid) & (regimes.trial_id == tid)
                & (regimes.episode == epi)]
    display(r[["run", "regime", "quadrant_type", "duration_s", "theta_start_deg",
               "theta_end_deg", "max_abs_theta_deg", "pct_I", "pct_CR",
               "pct_A", "pct_D"]])

P001 / T001 / episode 0


,duration_s,theta0_deg,T0_s,duration_over_T0,fall_cause
0,1.917,5.863,2.35,0.816,angle


,run,regime,quadrant_type,duration_s,theta_start_deg,theta_end_deg,max_abs_theta_deg,pct_I,pct_CR,pct_A,pct_D
0,0,Saved,fall,1.4334,5.8628,19.1979,19.1979,95.35,4.65,0.0,0.0
1,1,Safe,safe,0.2333,19.2772,1.3829,19.2772,0.00,100.00,0.0,0.0
2,2,Failed,fall,0.2500,-1.5233,-61.7364,61.7364,0.00,0.00,0.0,100.0


## Cikti

In [11]:
df_built.drop(columns=["state_defined"], errors="ignore").to_parquet(
    INTERIM_DIR / "samples_built.parquet", index=False)
episodes.to_parquet(INTERIM_DIR / "episodes.parquet", index=False)
regimes.to_parquet(INTERIM_DIR / "regimes.parquet", index=False)
input_events.to_parquet(INTERIM_DIR / "input_events.parquet", index=False)

print(f"samples_built.parquet  ({len(df_built):,} satir)")
print(f"episodes.parquet       ({len(episodes):,} satir)")
print(f"regimes.parquet        ({len(regimes):,} satir)")
print(f"input_events.parquet   ({len(input_events):,} satir)")

samples_built.parquet  (623,700 satir)
episodes.parquet       (1,332 satir)
regimes.parquet        (15,133 satir)
input_events.parquet   (28,488 satir)
